# Step 6 — Consume the deployed YOLO endpoint

The platform team has deployed our registered ONNX model behind **OpenVINO Model Server (OVMS)** over the KServe v2 protocol. We POST each of the 10 held-out images in [`datasets/ppe/images/test/`](datasets/ppe/images/test/) — images the VLM never saw and the YOLO model was never trained on — and render the predicted boxes.

Because OVMS serves the raw ONNX graph, the YOLO pre/post-processing that Ultralytics usually does for you moves **client-side**:

- **Preprocess**: letterbox-resize to 640×640, normalize to `[0,1]` float32, transpose HWC → NCHW, send as an `FP32` tensor under input name `"images"`.
- **Postprocess**: response is shape `[1, 11, num_anchors]` — 4 bbox coords + 7 class scores per anchor. We apply confidence threshold, argmax, NMS, and rescale boxes from 640×640 model space back to the original image.

This closes the loop: raw images → VLM annotations → trained YOLO → registered + deployed on RHOAI → callable over HTTPS from any production workload that needs real-time PPE detection.

In [ ]:
%%capture
!pip install requests pillow matplotlib numpy

In [ ]:
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import requests
from PIL import Image, ImageDraw, ImageFont

In [ ]:
# Endpoint default points at the cluster-internal predictor Service on port 8888
# (OVMS's REST port). The Service is HEADLESS so port 80 won't work — clients
# must target the container port directly. If the platform team exposed an
# external Route instead, paste the full https://... into PPE_INFERENCE_ENDPOINT.
ENDPOINT = os.environ.get(
    "PPE_INFERENCE_ENDPOINT",
    "http://ppe-yolov8n-vlm-2-predictor.ppe-detection-cv.svc.cluster.local:8888",
)

AUTH_TOKEN   = os.environ.get("PPE_INFERENCE_TOKEN")
MODEL_NAME   = os.environ.get("PPE_MODEL_NAME", "ppe-yolov8n-vlm-2")
INPUT_SIZE   = 640                  # YOLOv8 was exported at 640×640
CONF_THRESH  = 0.25                 # confidence gate for a detection
IOU_THRESH   = 0.45                 # NMS IoU threshold
CLASSES      = ["person", "helmet", "no-helmet", "vest", "no-vest", "gloves", "no-gloves"]
LABEL_COLORS = {
    "person":     (70,  130, 200),
    "helmet":     (0,   200, 0),
    "no-helmet":  (220, 20,  60),
    "vest":       (50,  205, 50),
    "no-vest":    (255, 69,  0),
    "gloves":     (32,  178, 170),
    "no-gloves":  (199, 21,  133),
}

print("Endpoint:  ", ENDPOINT)
print("Model:     ", MODEL_NAME)
print("Input size:", f"{INPUT_SIZE}x{INPUT_SIZE}")

# Smoke test: does the endpoint report the model as ready?
r = requests.get(
    f"{ENDPOINT.rstrip('/')}/v2/models/{MODEL_NAME}/ready",
    headers={"Authorization": f"Bearer {AUTH_TOKEN}"} if AUTH_TOKEN else {},
    timeout=10,
)
print(f"\n/v2/models/{MODEL_NAME}/ready  → HTTP {r.status_code}")
print(r.text[:300] if r.text else "(empty body)")

## Preprocess and call the endpoint

Same preprocessing the `YOLO().predict()` API does internally:

1. **Letterbox** to 640×640, padding with gray (114,114,114) to preserve aspect ratio.
2. Convert RGB → float32, divide by 255.
3. Transpose HWC → CHW, add batch dim → shape `[1, 3, 640, 640]`.
4. Flatten the float array and POST it as KServe v2 `infer` payload with input name `"images"` and `datatype: "FP32"`.

We keep the letterbox scale factor `r` and pad offsets `(dw, dh)` so we can unwind them after inference to land back in original-image coordinates.

In [ ]:
def letterbox(img: Image.Image, new_shape: int = 640, color=(114, 114, 114)):
    """Resize with preserved aspect ratio, pad to square. Returns (padded, r, dw, dh)."""
    w0, h0 = img.size
    r = min(new_shape / h0, new_shape / w0)
    new_unpad = (int(round(w0 * r)), int(round(h0 * r)))
    dw = (new_shape - new_unpad[0]) / 2
    dh = (new_shape - new_unpad[1]) / 2
    resized = img.resize(new_unpad, Image.BILINEAR)
    padded = Image.new("RGB", (new_shape, new_shape), color)
    padded.paste(resized, (int(round(dw - 0.1)), int(round(dh - 0.1))))
    return padded, r, dw, dh


def preprocess(image_path: Path):
    img = Image.open(image_path).convert("RGB")
    padded, r, dw, dh = letterbox(img, INPUT_SIZE)
    arr = np.asarray(padded, dtype=np.float32) / 255.0      # HWC float32 [0,1]
    arr = arr.transpose(2, 0, 1)                             # CHW
    arr = np.ascontiguousarray(arr)[None, ...]               # NCHW, shape [1,3,640,640]
    return img, arr, r, dw, dh


def auth_headers() -> dict:
    h = {"Content-Type": "application/json"}
    if AUTH_TOKEN:
        h["Authorization"] = f"Bearer {AUTH_TOKEN}"
    return h


def build_payload(tensor: np.ndarray) -> dict:
    return {
        "inputs": [{
            "name":     "images",
            "shape":    list(tensor.shape),
            "datatype": "FP32",
            "data":     tensor.flatten().tolist(),
        }]
    }


def predict(tensor: np.ndarray) -> dict:
    url = f"{ENDPOINT.rstrip('/')}/v2/models/{MODEL_NAME}/infer"
    r = requests.post(url, json=build_payload(tensor), headers=auth_headers(), timeout=120)
    r.raise_for_status()
    return r.json()

In [ ]:
TEST_DIR = Path("datasets/ppe/images/test")
test_images = sorted(p for p in TEST_DIR.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"})
print(f"Held-out test images: {len(test_images)}")

# Smoke test on the first image.
img0, tensor0, r0, dw0, dh0 = preprocess(test_images[0])
response = predict(tensor0)

# OVMS returns: outputs=[{name:"output0", datatype:"FP32", shape:[1,11,N], data:[...flat...]}]
out0 = response["outputs"][0]
print(f"Output name : {out0['name']}")
print(f"Output dtype: {out0['datatype']}")
print(f"Output shape: {out0['shape']}  (batch, 4_bbox+7_classes, num_anchors)")
print(f"data length : {len(out0['data'])}  (= {np.prod(out0['shape'])})")

## Decode the YOLOv8 output tensor → `{label, bbox_2d, confidence}`

The response is a raw YOLOv8 detection head, shape `[1, 4+C, N]` with `C=7` classes and `N≈8400` candidate anchors. Per anchor:

- `[0:4]` = center-format bbox `(cx, cy, w, h)` in **640×640 model coordinates**.
- `[4:4+C]` = per-class scores (already sigmoid'd by the ONNX graph).

Steps:

1. Reshape + transpose to `[N, 11]`.
2. For each anchor: `conf = max(class_scores)`, `cls = argmax(class_scores)`.
3. Filter to rows where `conf >= CONF_THRESH`.
4. Convert `(cx,cy,w,h)` → `(x1,y1,x2,y2)`.
5. NMS.
6. Unwind the letterbox: subtract pad offsets, divide by scale `r`, clip to original image bounds.

In [ ]:
def _nms(boxes: np.ndarray, scores: np.ndarray, iou_threshold: float) -> list:
    """Greedy IoU-based NMS. boxes: [N,4] xyxy; scores: [N]. Returns kept indices."""
    if boxes.size == 0:
        return []
    x1, y1, x2, y2 = boxes[:, 0], boxes[:, 1], boxes[:, 2], boxes[:, 3]
    areas = (x2 - x1).clip(0) * (y2 - y1).clip(0)
    order = scores.argsort()[::-1]
    keep = []
    while order.size > 0:
        i = order[0]
        keep.append(int(i))
        if order.size == 1:
            break
        xx1 = np.maximum(x1[i], x1[order[1:]])
        yy1 = np.maximum(y1[i], y1[order[1:]])
        xx2 = np.minimum(x2[i], x2[order[1:]])
        yy2 = np.minimum(y2[i], y2[order[1:]])
        inter = np.maximum(0, xx2 - xx1) * np.maximum(0, yy2 - yy1)
        iou = inter / (areas[i] + areas[order[1:]] - inter + 1e-9)
        order = order[1:][iou < iou_threshold]
    return keep


def decode(response: dict, orig_size, r: float, dw: float, dh: float,
           conf_threshold: float = CONF_THRESH,
           iou_threshold:  float = IOU_THRESH) -> list:
    out   = response["outputs"][0]
    shape = out["shape"]                                              # [1, 11, N]
    data  = np.asarray(out["data"], dtype=np.float32).reshape(shape)  # [1, 11, N]
    preds = data[0].T                                                 # [N, 11]

    boxes_cxcywh = preds[:, :4]
    cls_scores   = preds[:, 4:4 + len(CLASSES)]
    conf         = cls_scores.max(axis=1)
    cls_ids      = cls_scores.argmax(axis=1)

    keep_mask = conf >= conf_threshold
    if not keep_mask.any():
        return []
    boxes_cxcywh = boxes_cxcywh[keep_mask]
    conf         = conf[keep_mask]
    cls_ids      = cls_ids[keep_mask]

    # (cx,cy,w,h) -> (x1,y1,x2,y2) in 640×640 model space
    cx, cy, w, h = boxes_cxcywh.T
    x1 = cx - w / 2; y1 = cy - h / 2
    x2 = cx + w / 2; y2 = cy + h / 2
    boxes_xyxy = np.stack([x1, y1, x2, y2], axis=1)

    # Per-class NMS so overlapping boxes of different classes both survive.
    keep_idx = []
    for c in np.unique(cls_ids):
        mask = cls_ids == c
        idx_in_class = np.where(mask)[0]
        kept_local   = _nms(boxes_xyxy[mask], conf[mask], iou_threshold)
        keep_idx.extend(idx_in_class[kept_local].tolist())
    if not keep_idx:
        return []
    boxes_xyxy = boxes_xyxy[keep_idx]
    conf       = conf[keep_idx]
    cls_ids    = cls_ids[keep_idx]

    # Unwind letterbox: subtract pad, divide by scale, clip to original image.
    boxes_xyxy[:, [0, 2]] -= dw
    boxes_xyxy[:, [1, 3]] -= dh
    boxes_xyxy /= r
    w0, h0 = orig_size
    boxes_xyxy[:, [0, 2]] = boxes_xyxy[:, [0, 2]].clip(0, w0)
    boxes_xyxy[:, [1, 3]] = boxes_xyxy[:, [1, 3]].clip(0, h0)

    return [
        {"label": CLASSES[int(c)], "bbox_2d": b.tolist(), "confidence": float(s)}
        for b, s, c in zip(boxes_xyxy, conf, cls_ids)
    ]


detections = decode(response, img0.size, r0, dw0, dh0)
print(f"Got {len(detections)} detections on {test_images[0].name}.")
for d in detections[:10]:
    print(" ", d)

## Run inference on all 10 test images and show a grid

In [ ]:
def draw(img: Image.Image, detections: list) -> Image.Image:
    img = img.copy()
    d = ImageDraw.Draw(img)
    try:
        font = ImageFont.truetype("/System/Library/Fonts/Helvetica.ttc", 14)
    except OSError:
        font = ImageFont.load_default()
    for det in detections:
        label = det["label"]
        x1, y1, x2, y2 = det["bbox_2d"]
        color = LABEL_COLORS.get(label, (0, 255, 0))
        d.rectangle([x1, y1, x2, y2], outline=color, width=3)
        text = f"{label} {det['confidence']:.2f}"
        tb = d.textbbox((0, 0), text, font=font)
        th = tb[3] - tb[1]
        d.rectangle([x1, max(0, y1 - th - 4), x1 + (tb[2] - tb[0]) + 6, y1], fill=color)
        d.text((x1 + 3, max(0, y1 - th - 2)), text, fill=(0, 0, 0), font=font)
    return img


OUTPUT_DIR = Path("output"); OUTPUT_DIR.mkdir(exist_ok=True)

fig, axes = plt.subplots(2, 5, figsize=(24, 10))
for ax, img_path in zip(axes.flat, test_images):
    img, tensor, r, dw, dh = preprocess(img_path)
    dets = decode(predict(tensor), img.size, r, dw, dh)
    annotated = draw(img, dets)
    annotated.save(OUTPUT_DIR / f"{img_path.stem}_deployed.jpg")
    ax.imshow(annotated)
    ax.set_title(f"{img_path.name} \u2014 {len(dets)} dets", fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

---

**Recap of the workshop story.**
1. [01] Raw, unlabeled images → loaded into FiftyOne.
2. [02] Qwen2.5-VL drew bounding boxes on them, zero-shot — no training data, no manual labeling.
3. [03] Exported to YOLO format.
4. [04] Fine-tuned a small, fast YOLOv8n.
5. [05] Registered in RHOAI Model Registry, deployed with KServe.
6. [06] Called the deployed endpoint from this notebook — this is what a production app would do.